# 07 · One model for all thresholds, and forecasting extreme bursts

Instead of fitting each threshold separately, fit a single model
to the waiting times of *all* thresholds at once:

* the **shape** k is shared by all thresholds (supported by notebooks
  `04`–`05`),
* the **scale** grows with the threshold as a power law,
  **scale(T) = b0 · T^b1**.

Because the scale is a formula in T, the model can be extrapolated to a
threshold that has (almost) never been reached, here 10⁶ SFU, and used to
estimate the probability of such a burst within the next 10 years.

**Steps**

1. Build one table with the waiting times of every threshold.
2. Fit the regression model for Weibull, Gamma and Exponential; compare AIC.
3. Forecast: unconditional and conditional probability of an extreme burst
   within the horizon (see `srb/forecasting.py` for the formulas).
4. Bootstrap the whole procedure to get uncertainties.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd

from srb.config import RANDOM_SEED, TABLES_DIR
from srb.data import inter_event_times_by_threshold, load_catalogue, select_thresholds, to_long_table
from srb.diagnostics import bootstrap_summary
from srb.distributions import EXPONENTIAL, GAMMA, WEIBULL
from srb.fitting import fit_scale_regression
from srb.forecasting import bootstrap_forecast, forecast_probabilities

## Parameters

In [2]:
START_THRESHOLD = 4000  # SFU; adjust depending on the spectral types (ST) included
END_THRESHOLD = 100000  # SFU
FORECAST_THRESHOLD = 1e6  # SFU
FORECAST_YEARS = 10
FORECAST_START_TIME = pd.Timestamp.today()  # the conditional probability depends on this date
BOOTSTRAP_REPEATS = 10000
MODELS = [WEIBULL, GAMMA, EXPONENTIAL]

HORIZON_DAYS = FORECAST_YEARS * 365

## 1. Joint table of waiting times


In [3]:
df = load_catalogue()
groups = inter_event_times_by_threshold(df, select_thresholds(START_THRESHOLD, END_THRESHOLD))
long_table = to_long_table(groups)
long_table.to_csv(TABLES_DIR / "joint_interoccurrence_times.csv", index=False)

print("Thresholds used:")
print(np.array(list(groups)))
print(f"\nTotal rows in joint table: {len(long_table)}")
long_table.head()

Skipping T = 100000.00: only 6 inter-event times
Thresholds used:
[ 4281.33239872  5455.59478117  6951.92796178  8858.6679041
 11288.37891685 14384.49888288 18329.80710832 23357.2146909
 29763.51441631 37926.90190732 48329.30238572 61584.8211066
 78475.99703515]

Total rows in joint table: 949


,threshold,inter_event_days
0,4281.332399,107.237361
1,4281.332399,524.738333
2,4281.332399,0.122847
3,4281.332399,23.404792
4,4281.332399,95.335417


## 2. Fit the regression model and compare AIC

In [4]:
fits = {dist.name: fit_scale_regression(long_table, dist) for dist in MODELS}

for name, fit in fits.items():
    print(f"\nFitted {name} regression:")
    if len(fit.shape_params):
        print(f"shape = {fit.shape_params[0]:.4f}")
    print(f"b0    = {fit.b0:.4f}")
    print(f"b1    = {fit.b1:.4f}")
    print(f"scale = {fit.b0:.4f} * T^{fit.b1:.4f}")
    print(f"NLL   = {fit.nll:.4f}")
    print(f"AIC   = {fit.aic:.4f}")

print(f"\nLowest AIC: {min(fits, key=lambda name: fits[name].aic)}")


Fitted Weibull regression:
shape = 0.3698
b0    = 0.0041
b1    = 1.0729
scale = 0.0041 * T^1.0729
NLL   = 5023.5780
AIC   = 10053.1561

Fitted Gamma regression:
shape = 0.2571
b0    = 0.3492
b1    = 0.8530
scale = 0.3492 * T^0.8530
NLL   = 4977.5692
AIC   = 9961.1384

Fitted Exponential regression:
b0    = 0.0898
b1    = 0.8530
scale = 0.0898 * T^0.8530
NLL   = 6096.1132
AIC   = 12196.2264

Lowest AIC: Gamma


## 3. Forecast

* **Unconditional**: probability that the waiting time to the next burst
  above `FORECAST_THRESHOLD` is shorter than the horizon.
* **Conditional**: the same, but taking into account that no such burst
  has happened since the last one in the catalogue.

Since there's evidence that there was one exceeding the $$10^6 \text{ SFU,}$$ the proper probability is the conditional.

In [5]:
forecasts = {
    name: forecast_probabilities(df, fit, FORECAST_THRESHOLD, HORIZON_DAYS, FORECAST_START_TIME)
    for name, fit in fits.items()
}

print(f"Forecast threshold = {FORECAST_THRESHOLD:.0f}")
print(f"Forecast period    = {FORECAST_YEARS} years")
for name, forecast in forecasts.items():
    print(f"{name + ' forecast scale':27s}= {forecast['scale']:.4f} days")

any_forecast = next(iter(forecasts.values()))
if any_forecast["last_event_time"] is None:
    print(f"No event above {FORECAST_THRESHOLD:.0f} was found in the data.")
else:
    print(f"Latest event above {FORECAST_THRESHOLD:.0f}: {any_forecast['last_event_time']}")
    print(f"Time since latest event: {any_forecast['elapsed_days']:.2f} days")

Forecast threshold = 1000000
Forecast period    = 10 years
Weibull forecast scale     = 11315.8249 days
Gamma forecast scale       = 45848.5084 days
Exponential forecast scale = 11786.4284 days
Latest event above 1000000: 2006-12-06 18:42:00
Time since latest event: 7286.71 days


## 4. Bootstrap uncertainty

Resample the rows of the joint table, refit, and recompute both
probabilities `BOOTSTRAP_REPEATS` times per model. This is the slow step.

In [6]:
rng = np.random.default_rng(RANDOM_SEED)
bootstrap = {}
for dist in MODELS:
    print(f"Bootstrapping {dist.name}...")
    bootstrap[dist.name] = bootstrap_forecast(
        long_table, df, dist, FORECAST_THRESHOLD, HORIZON_DAYS, FORECAST_START_TIME, BOOTSTRAP_REPEATS, rng
    )

Bootstrapping Weibull...
Bootstrapping Gamma...
Bootstrapping Exponential...


In [7]:
rows = []
for name, forecast in forecasts.items():
    p_uncond_boot, p_cond_boot = bootstrap[name]
    for kind, p, values in [
        ("unconditional", forecast["p_unconditional"], p_uncond_boot),
        ("conditional", forecast["p_conditional"], p_cond_boot),
    ]:
        error, low, high = bootstrap_summary(values)
        rows.append({
            "model": name,
            "probability": kind,
            "p": p,
            "bootstrap_error": error,
            "ci95_low": low,
            "ci95_high": high,
        })

forecast_table = pd.DataFrame(rows)
forecast_table.round(6)

,model,probability,p,bootstrap_error,ci95_low,ci95_high
0,Weibull,unconditional,0.482177,0.050644,0.390547,0.586914
1,Weibull,conditional,0.128616,0.018120,0.098769,0.168988
2,Gamma,unconditional,0.567364,0.047501,0.480687,0.666942
3,Gamma,conditional,0.187110,0.034035,0.135969,0.269703
4,Exponential,unconditional,0.266317,0.077188,0.151014,0.451199
5,Exponential,conditional,0.266317,0.077188,0.151014,0.451199


`p` is the probability of at least one burst above `FORECAST_THRESHOLD` within `FORECAST_YEARS` years. For the exponential the conditional and unconditional values coincide (no memory); for Weibull and Gamma with shape < 1, the longer we have waited, the lower the conditional probability.

The model with the smallest AIC was the Gamma distribution. Therefore, the final proability is $$\boxed{(18.7\pm3.4)\%.}$$